# Interpreting Logistic Regression Coefficients: Log-Odds, Odds Ratios, and Marginal Effects

## 1. Introduction and Overview

In classification models, predicting a probability is only half the battle. In many domains like healthcare, finance, and regulatory environments, understanding exactly how a feature impacts the outcome is legally and scientifically required.

In a logistic regression model, the estimated coefficients (beta) do NOT represent the change in the probability of the outcome for a unit change in the predictor. Instead, they represent the change in the log-odds of the outcome. To communicate model results effectively, coefficients must be transformed into Odds Ratios (OR) or Average Marginal Effects (AME).

This notebook provides a rigorous technical analysis of coefficient interpretation, detailing the mathematical transition from linear probability to log-odds, deriving the formulas for odds ratios, and establishing standard engineering practices for business interpretation.

In [ ]:
# Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings

# Configure environment
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for complete reproducibility
np.random.seed(42)
print('Libraries imported successfully and random seed set.')

## 2. Probability vs. Odds

Before defining the Odds Ratio, we must formally separate Probability from Odds.

- Probability Space: Bounded between 0 and 1. (e.g., 80 percent chance of rain -> p = 0.8)
- Odds Space: The ratio of the probability of success to the probability of failure. Bounded between 0 and infinity.

Formula:
Odds = p / (1 - p)

If p = 0.8, the odds are 0.8 / 0.2 = 4. We say the odds are 4 to 1.

In [ ]:
def prob_to_odds(p):
    return p / (1 - p)

def odds_to_prob(o):
    return o / (1 + o)

test_prob = 0.80
test_odds = prob_to_odds(test_prob)

print(f'If Probability is {test_prob:.2f}:')
print(f'The Odds are {test_odds:.2f} (or 4 to 1)')
print(f'Converting back yields a probability of {odds_to_prob(test_odds):.2f}')

## 3. Synthetic Data Creation: Loan Default

To illustrate coefficient interpretation, we will create a synthetic dataset representing a bank's loan portfolio. The target is to predict loan default (1 = Default, 0 = Paid) based on Age, Credit Score, and Number of Late Payments.

We simulate this strictly within the notebook using an underlying log-odds formulation to ensure we know the true data-generating process.

In [ ]:
def generate_loan_data(n_samples=2000):
    age = np.random.normal(45, 10, n_samples)
    credit_score = np.random.normal(680, 60, n_samples)
    late_payments = np.random.poisson(1.2, n_samples)
    
    # True log-odds generating function
    # Age and credit score decrease risk, late payments increase risk
    true_log_odds = 5.0 - 0.02 * age - 0.01 * credit_score + 0.8 * late_payments
    
    # Convert to probabilities using Sigmoid
    probabilities = 1 / (1 + np.exp(-true_log_odds))
    default = np.random.binomial(1, probabilities)
    
    df = pd.DataFrame({
        'default': default,
        'age': np.round(age, 1),
        'credit_score': np.round(credit_score, 0),
        'late_payments': late_payments
    })
    return df

df_loans = generate_loan_data()
print('Synthetic Loan Data Generated.')
print(df_loans.head())

## 4. The Log-Odds Space (Raw Coefficients)

In logistic regression, the relationship between the predictors X and the probability P(Y=1|X) is non-linear. The model posits that the log-odds (logit) of the probability is a linear combination of the predictors:

ln(p / (1 - p)) = beta_0 + beta_1 * X_1 + beta_2 * X_2 ...

Let us fit a logistic regression model using statsmodels and examine these raw coefficients.

In [ ]:
features = ['age', 'credit_score', 'late_payments']
X = sm.add_constant(df_loans[features])
y = df_loans['default']

# Fit the Logistic Regression Model via Maximum Likelihood Estimation
logit_model = sm.Logit(y, X).fit(disp=0)

raw_coefs = logit_model.params
print('--- Raw Coefficients (Log-Odds) ---')
print(raw_coefs.round(4))
print('')
print('Interpretation for late_payments:')
print('For every additional late payment, the log-odds of loan default increase by approx 0.81.')
print('This log-odds value is mathematically true but completely unintuitive for business stakeholders.')

## 5. The Odds Ratio (Multiplicative Interpretation)

By exponentiating both sides of the logit equation, we isolate the odds:
Odds = p / (1 - p) = exp(beta_0 + beta_1 * X_1 + ...)

If we increase a continuous predictor X_j by one unit, the new odds multiply the old odds by exp(beta_j). Therefore, exp(beta_j) is the Odds Ratio (OR). It represents the multiplicative factor by which the odds of the outcome change for a one-unit increase in X_j.

In [ ]:
# Calculate Odds Ratios and 95% Confidence Intervals
odds_ratios = np.exp(raw_coefs)
conf_int = logit_model.conf_int()
or_conf_int = np.exp(conf_int)

or_df = pd.DataFrame({
    'Log-Odds (Beta)': raw_coefs,
    'Odds Ratio': odds_ratios,
    'CI_Lower_95': or_conf_int[0],
    'CI_Upper_95': or_conf_int[1]
})

print('--- Odds Ratios Interpretation Table ---')
print(or_df.round(4))
print('')
late_pymt_or = or_df.loc['late_payments', 'Odds Ratio']
print('Business Interpretation for Late Payments:')
print(f'An Odds Ratio of {late_pymt_or:.2f} means that each additional late payment multiplies the odds of default by {late_pymt_or:.2f}.')
print(f'Equivalently, it increases the odds of default by {((late_pymt_or - 1) * 100):.1f} percent.')

## 6. Visualizing Odds Ratios: The Forest Plot

A standard engineering practice for presenting Odds Ratios is a Forest Plot. The vertical line at OR = 1.0 represents no effect. Variables to the right increase the odds of the event; variables to the left decrease the odds.

In [ ]:
plot_df = or_df.drop('const')
variables = plot_df.index
or_values = plot_df['Odds Ratio']
lower_errors = or_values - plot_df['CI_Lower_95']
upper_errors = plot_df['CI_Upper_95'] - or_values

plt.figure(figsize=(10, 5))
plt.errorbar(x=or_values, y=variables, xerr=[lower_errors, upper_errors], 
             fmt='o', color='darkred', markersize=10, capsize=5, linewidth=2)

plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=2, alpha=0.7)
plt.title('Odds Ratios with 95% Confidence Intervals')
plt.xlabel('Odds Ratio (exp(Beta)) | <-- Decreases Risk | Increases Risk -->')
plt.ylabel('Predictor Variables')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Average Marginal Effects (AME)

While Odds Ratios are excellent for understanding relative multipliers, humans intuitively think in probabilities (e.g., What is the percentage point increase in risk?).

To find the actual change in probability, we take the partial derivative of p with respect to X_j:
dp / dX_j = beta_j * p * (1 - p)

Because this varies based on p, we compute the Average Marginal Effect (AME) across all actual observations in the dataset.

In [ ]:
# Calculate Average Marginal Effects (AME) using statsmodels
margeff = logit_model.get_margeff(at='overall', method='dydx')
print(margeff.summary())
print('')

ame_values = margeff.margeff
ame_names = features

print('--- Additive Probability Interpretations (AME) ---')
for name, ame in zip(ame_names, ame_values):
    print(f'{name}: A 1-unit increase changes the probability of default by {(ame*100):.2f} percentage points on average.')

## 8. Visualizing the Transformation Pipeline

Let us visualize the three spaces: Log-Odds (Linear), Odds (Exponential), and Probability (Sigmoid). This demonstrates why a constant log-odds coefficient creates non-linear probability changes.

In [ ]:
x_vals = np.linspace(-2, 10, 200)
b0, b1 = -4.0, 0.8

log_odds_space = b0 + b1 * x_vals
odds_space = np.exp(log_odds_space)
prob_space = 1 / (1 + np.exp(-log_odds_space))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(x_vals, log_odds_space, color='blue', linewidth=3)
axes[0].set_title('1. Log-Odds Space (Linear)')
axes[0].set_xlabel('Predictor X')
axes[0].set_ylabel('Log-Odds')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_vals, odds_space, color='green', linewidth=3)
axes[1].set_title('2. Odds Space (Exponential)')
axes[1].set_xlabel('Predictor X')
axes[1].set_ylabel('Odds')
axes[1].grid(True, alpha=0.3)

axes[2].plot(x_vals, prob_space, color='red', linewidth=3)
axes[2].set_title('3. Probability Space (Sigmoid)')
axes[2].set_xlabel('Predictor X')
axes[2].set_ylabel('Probability P(Y=1|X)')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Non-Linear Impact on Absolute Probability

An Odds Ratio of 2.0 has a massive impact on absolute probability if your baseline probability is near 0.5. However, it has a tiny impact if your baseline probability is 0.98.

In [ ]:
baseline_probs = np.linspace(0.01, 0.99, 100)
baseline_odds = baseline_probs / (1 - baseline_probs)

constant_OR = 2.0
new_odds = baseline_odds * constant_OR
new_probs = new_odds / (1 + new_odds)

prob_diff = new_probs - baseline_probs

plt.figure(figsize=(8, 5))
plt.plot(baseline_probs, prob_diff, color='purple', linewidth=3)
plt.title('Absolute Change in Probability given an Odds Ratio of 2.0')
plt.xlabel('Baseline Probability (Before applying OR)')
plt.ylabel('Absolute Increase in Probability')
plt.axvline(0.5, color='gray', linestyle='--', label='Max impact at P=0.5')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 10. Machine Learning Connections: Scikit-Learn

In machine learning, logistic regression is often treated purely as a classification algorithm. The weights of the network are exactly the beta coefficients. We must disable regularization to match standard statistical outputs.

In [ ]:
# penalty=None disables L2 regularization
sklearn_model = LogisticRegression(penalty=None, fit_intercept=True)
sklearn_model.fit(df_loans[features], y)

print('--- Framework Comparison ---')
print('Statsmodels (MLE) Coefficients:')
print(raw_coefs.drop('const').round(4))
print('')
print('Scikit-Learn (Binary Cross-Entropy) Coefficients:')
for feat, coef in zip(features, sklearn_model.coef_[0]):
    print(f'{feat}: {coef:.4f}')
print('')
print('Conclusion: Machine Learning Binary Cross-Entropy is mathematically identical to Statistical MLE.')

## 11. Practical Example: A/B Testing

Let us apply this to a real-world scenario. An e-commerce platform tests a new checkout UI. The treatment group receives the new UI (1), control gets the old UI (0).

In [ ]:
n_ab = 3000
treatment = np.random.binomial(1, 0.5, n_ab)
user_age = np.random.normal(35, 12, n_ab)

lo_ab = -2.5 + 0.223 * treatment + 0.01 * user_age
converted = np.random.binomial(1, 1 / (1 + np.exp(-lo_ab)))

df_ab = pd.DataFrame({'converted': converted, 'treatment': treatment, 'user_age': user_age})
X_ab = sm.add_constant(df_ab[['treatment', 'user_age']])
ab_model = sm.Logit(df_ab['converted'], X_ab).fit(disp=0)

trt_beta = ab_model.params['treatment']
trt_or = np.exp(trt_beta)

print('--- A/B Test Engineering Report ---')
print(f'Log-Odds Coefficient: {trt_beta:.4f}')
print(f'Odds Ratio: {trt_or:.4f}')
print(f'Business Interpretation: Users exposed to the new checkout UI have {(trt_or - 1)*100:.1f} percent higher odds of converting compared to the control group.')

## 12. Feature Standardization Impact

If you apply StandardScaler to your features, a 1-unit increase no longer means 1 raw unit. It means 1 standard deviation. You must alter your interpretation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_loans[['age', 'credit_score']])

clf_scaled = LogisticRegression(penalty=None)
clf_scaled.fit(X_scaled, y)

scaled_or_age = np.exp(clf_scaled.coef_[0][0])
std_dev_age = df_loans['age'].std()

print('--- Interpreting Standardized Features ---')
print(f'The standardized Odds Ratio for Age is {scaled_or_age:.4f}.')
print(f'Interpretation: For every 1 standard deviation increase in Age ({std_dev_age:.2f} years),')
print(f'the odds of default are multiplied by {scaled_or_age:.4f}.')

## 13. Practice Exercise: Public Health Model

Scenario: A public health model assesses the probability of developing a disease based on BMI and Smoker status (1=Yes, 0=No).

Task:
1. Fit a logistic regression model on the generated dataset.
2. Calculate the Odds Ratio for being a Smoker.

In [ ]:
# Generate Exercise Data
n_ex = 1500
bmi = np.random.normal(28, 5, n_ex)
smoker = np.random.binomial(1, 0.25, n_ex)
ex_log_odds = -6.0 + 0.15 * bmi + 1.386 * smoker
disease = np.random.binomial(1, 1 / (1 + np.exp(-ex_log_odds)))

df_ex = pd.DataFrame({'disease': disease, 'bmi': bmi, 'smoker': smoker})
print('Exercise data generated. Target: disease. Features: bmi, smoker')

### Solution

In [ ]:
X_ex = sm.add_constant(df_ex[['bmi', 'smoker']])
y_ex = df_ex['disease']

ex_model = sm.Logit(y_ex, X_ex).fit(disp=0)
smoker_beta = ex_model.params['smoker']
smoker_or = np.exp(smoker_beta)

print(f'The Odds Ratio for smoking is {smoker_or:.2f}.')
print(f'Interpretation: Smokers have {smoker_or:.2f} times the odds of developing the disease compared to non-smokers.')

## 14. Visualization Gallery: 2D Probability Contour

To fully understand interaction in a multi-variable logistic model, we can plot a 2D contour map. This shows how probability changes jointly across two continuous variables (Age and Credit Score) from our original loan dataset.

In [ ]:
age_grid = np.linspace(df_loans['age'].min(), df_loans['age'].max(), 100)
credit_grid = np.linspace(df_loans['credit_score'].min(), df_loans['credit_score'].max(), 100)
xx, yy = np.meshgrid(age_grid, credit_grid)

grid_const = np.ones_like(xx.ravel())
grid_late = np.zeros_like(xx.ravel())

grid_X = np.column_stack((grid_const, xx.ravel(), yy.ravel(), grid_late))
Z = logit_model.predict(grid_X).reshape(xx.shape)

plt.figure(figsize=(10, 6))
contour = plt.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.8)
plt.colorbar(contour, label='Probability of Default P(Y=1|X)')
plt.title('Probability Contour: Age vs Credit Score (Holding Late Payments = 0)')
plt.xlabel('Age')
plt.ylabel('Credit Score')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 15. Summary and Key Takeaways

- Log-Odds (The Raw Beta): Logistic regression coefficients represent the change in log-odds. They are strictly additive and linear, but highly unintuitive for human interpretation.
- Odds Ratios (exp(Beta)): Provide a multiplicative interpretation. A 1-unit increase in X multiplies the odds by exp(Beta). This is the standard in epidemiological and business reporting.
- Marginal Effects (dp/dx): Provide an additive, probability-based interpretation. Because the slope of the sigmoid curve changes, marginal effects depend on where you are on the curve.
- Average Marginal Effect (AME): The industry standard for computing the overall expected impact of a variable on the actual probability of the outcome. Computed by averaging the marginal effect across all actual observations.

In [ ]:
print('Notebook execution complete. Parameter interpretation module verified successfully.')